# Chapter 7. ??? ??? ?? ????

In [ ]:
                    import os
import numpy as np
np.random.seed(42)

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
matplotlib.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] = False

---
+ **??? ??**<sup>wisdom of the crowd</sup> : ???? ??? ?? ?? ???? ??? ??? ?? ??? ?? ?
+ **??? ??**<sup>Ensemble Learing</sup> : ?? ?? ??? ???? ? ???? ?????? ??? ??? ???? ? ?? ?? ???? ?
    + **???** : ??? ???
    + **??? ??**<sup>Ensemble method</sup> : ??? ?? ????
+ ?? ??? ??? ? **?? ????**<sup>Random Forest</sup>

---
## 7.1 ?? ?? ???

![Figure7-1](./images/Figure7-1.png)
**<center>?? 7-1 ?? ??? ?????</center>**

+ **?? ??**<sup>hard voting</sup> ??? : ??? ??? ???? ??? ???

![Figure7-2](./images/Figure7-2.png)
**<center>?? 7-2 ?? ?? ???? ??</center>**

+ ??? ?? ???? ???? ??? ?? ??? ? ?? ??? ???? ???? ?? ??? ??
+ ? ???? **?? ???**<sup>weak learner</sup>(?, ?? ???? ?? ? ?? ??? ?? ???)???? ???? ?? ????? ???? (?? ???? ??) **?? ???**<sup>strong learner</sup>? ? ? ??

![Figure7-3](./images/Figure7-3.png)
**<center>?? 7-3 ? ?? ??</center>**

+ ??? ??? ???? ??? ? ?? ????? ??? ????? ?? ? ??? ??? ???
+ ??? ???? ?? ??? ?? ?? ?????? ????? ?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
X_train.shape, y_train.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

log_clf = LogisticRegression(solver='liblinear', random_state=42)
rnd_clf = RandomForestClassifier(n_estimators=10, random_state=42)
svm_clf = SVC(gamma='auto', probability=True, random_state=42)

hard_voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting='hard')
hard_voting_clf.fit(X_train, y_train)

In [ ]:
soft_voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting='soft')
soft_voting_clf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

for clf in (log_clf, rnd_clf, svm_clf, hard_voting_clf, soft_voting_clf):
    clf.fit(X_train, y_train)
    y_pred_voting = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred_voting))

+ **?? ??**<sup>soft voting</sup> : ???? ?? ???? ? ?? ??? ?

---
## 7.2 ??? ????

+ ??? ???? ??? ??
    1. ?? ?? ?? ????? ??
    2. **?** ?? ????? ????? **?? ??? ???? ???? ??**
+ **??**<sup>bagging(bootstrap aggregating)</sup> : ?? ???? ??? ???? ????? ??
+ **????**<sup>pasting</sup> : ?? ???? ??? ???? ?? ????? ??

![Figure7-4](./images/Figure7-4.png)
**<center>?? 7-4 ????/??? ?? ?? ???? ??</center>**

+ **????** : ?? ???? ??? ??
    + ???? ?? ? **??? ???**<sup>statistical mode</sup>(e.g. ?? ?? ???)
    + ?? ? ??
+ ?? ???? ?? ?? ??? ???? ??? ?? ?? ???? ??? **?? ??? ???? ??? ??? ?? ??**
+ ????? ???? ??? ?? ?????? ??? ???? ???? ?? ??? **??? ????? ??? ???**

### 7.2.1 ????? ??? ????

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=500,
    max_samples=100, bootstrap=True, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)
y_pred_bag = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred_bag)

In [ ]:
pas_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=500,
    max_samples=100, bootstrap=False, n_jobs=-1, random_state=42)
pas_clf.fit(X_train, y_train)
y_pred_pas = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred_pas)

![Figure7-5](./images/Figure7-5.png)
**<center>?? 7-5 ?? ?? ??? 500? ??? ?? ?? ??? ??</center>**

+ **???? ??? ???? ? ?? ??? ??** - ?? ??? ?? ?? ?? ????? ?? ??? ? ???
+ ????? ??? ? ?? ??? ??? ??? ????? ? ????? ??? CPU ??? ??? ??? ?? ???? ??? ????? ?? ????? ?? ??

### 7.2.2 oob ??

+ **oob**<sup>out-of-bag</sup> ?? : ? ???? ???? ?? ?? ??? ???
    + ????? oob ??? ?? ??
    + ???? ???? ???? oob ??? ???? ???? **oob ??? ??? ??**? ??
    + ???? ??? ? ???? oob ??? ??

In [ ]:
oob_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=500,
    bootstrap=True, n_jobs=-1, oob_score=True, random_state=40)
oob_clf.fit(X_train, y_train)
oob_clf.oob_score_

In [ ]:
y_pred_oob = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred_oob)

In [ ]:
oob_clf.oob_decision_function_.shape # oob ??? ??? ? ?? ??? ??? ??

---
## 7.3 ?? ??? ?? ??????

+ **?? ???** - ? ??? ???? ??? ??? ??? ?? ??? ??
    + ???? ?? ?? ???? ????? ?? ? ???
+ **?? ?? ??**<sup>Random Patches method</sup> - ?? ??? ??? ?? ????? ??
+ **?? ?????? ??**<sup>Random Subspaces method</sup>
    + ?? ??? ?? ?? ? bootstrap=False, max_samples=1.0
    + ??? ??? ? bootstrap_features=True, max_features<1.0

---
## 7.4 ?? ????

+ ?? ????? ????? ?? ?? ????? ??? ?? ??? ???
+ ????? max_samples? ?? ??? ??? ??

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1, random_state=42)
rnd_clf.fit(X_train, y_train)
y_pred_rf = rnd_clf.predict(X_test)

accuracy_score(y_test, y_pred_rf)

+ ?? ????? ??? ??? ? ?? ?? ??? ??? ??? ?? ??, **???? ??? ?? ?? ??? ??? ??? ?? ?**?? ????? ? ??  
? ?? ?? ??? ?? ?? ?? ??? ??? ????? ? ??? ??? ??

In [ ]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(splitter="random", max_leaf_nodes=16, random_state=42),
    n_estimators=500, max_samples=1.0, bootstrap=True, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)
y_pred_bag = bag_clf.predict(X_test)

accuracy_score(y_test, y_pred_bag)

### 7.4.1 ???? ??

+ **???? ?? ??**<sup>Extremly Randomized Trees</sup> ???(?? ??? **???? ??**<sup>Extra-Trees</sup>) : ????? ???? ??? ?? ????
    + ???? ?? ??? ????? ?? ??? ??
    + ?? ???? ???? ?? ??? ???? ?? ?? ?? ?????? ?? ??? ?? ???? ?? ? ????? ???? ?? ?????? ???? ??? ?? ??

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

ext_clf = ExtraTreesClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1, random_state=42)
ext_clf.fit(X_train, y_train)
y_pred_ext = ext_clf.predict(X_test)

accuracy_score(y_test, y_pred_ext)

+ RandomForestClassifier? ExtraTreesClassifier ? ?? ? ??? ???? ?????, ????? ? ? ????? ?? ???? ????? ?? ??? ??

### 7.4.2 ?? ???

+ ?? ????? ??? ???? ? ? ?? ??? ???? ??? ??? ? ?? ???

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
rnd_clf = RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)
rnd_clf.fit(iris["data"], iris["target"])
for name, score in zip(iris["feature_names"], rnd_clf.feature_importances_):
    print(name, score)

![Figure7-6](./images/Figure7-6.png)
**<center>?? 7-6 (?? ???? ????? ??) MNIST ?? ???</center>**

---
## 7.5 ???

+ **???**<sup>boosting</sup>(??? **?? ???**<sup>hypothesis boosting</sup>?? ??) - ?? ???? ?? ? ???? ?? ???? ??? ??? ??

### 7.5.1 ?????

+ ?? ??? ?????? ?? ??? ???? ? ??? ??

![Figure7-7](./images/Figure7-7.png)
**<center>?? 7-7 ??? ???? ??????? ????? ???? ?????</center>**

![Figure7-8](./images/Figure7-8.png)
**<center>?? 7-8 ??? ???? ?? ??</center>**

+ ?? ???? ??? ??? ???? ????? ??? ???? ??? ??
+ ???? ???? ?? ????? ?? ???? ???

![Equation7-1](./images/Equation7-1.png)
**<center>? 7-1 j?? ???? ???? ??? ???</center>**

+ ? ?? ??? $w^{(i)}$? ??? $1 \over m$? ????

![Equation7-2](./images/Equation7-2.png)
**<center>? 7-2 ??? ???</center>**

+ $\eta$ : ??? ????(???? 1)
+ ???? ????? ???? ???? ? ???

![Equation7-3](./images/Equation7-3.png)
**<center>? 7-3 ??? ??? ???? ??</center>**
+ ?? ??? ??? ???? ???

#### ? ?????? ?? ??
1. ???? ????, ??? ???? ??? ???? ???? ??
2. ???? ???? ???? ???? ??
3. ?? ??? ???? ?????? ???
4. ?????? ???? ??? ??? ??? ???? ??

**?** ??? ??? ?? ????? ??? ???? ????? ???

![Equation7-4](./images/Equation7-4.png)
**<center>? 7-4 ????? ??</center>**

+ $N$? ???? ?
+ ???? ??? ?? ?? ? ???? ?? ??? ?

+ ????? SAMME?? ?????? ?? ??? ??? ??
+ ???? ? ? ?? ?? SAMME? ?????? ??
+ ???? ???? ??? ??? ? ???(?, predict_proba() ???? ???) ????? SAMME.R??? ??? ??
+ SAMME.R? ??? ?? ??? ??? ???? ????? ??? ? ??

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=200,
    algorithm="SAMME.R", learning_rate=0.5, random_state=42)
ada_clf.fit(X_train, y_train)

+ ???? ???? ?? ??? max_depth=1?, ?? ?? ??? ?? ?? ? ?? ???? ??
+ ? ??? AdaBoostClassifier? ?? ???

### 7.5.2 ????? ???

+ **Gradient Boosting** : ??????? ???? ??? ???? ???? ?? ?? ???? ?? **?? ??**<sup>residual error</sup>? ??? ???? ????
+ **Gradient Tree Boosting** or **Gradient Boosted Regression Tree**(GBRT) : ?? ??? ?? ???? ??

In [ ]:
np.random.seed(42)
X = np.random.rand(100, 1) - 0.5
y = 3*X[:, 0]**2 + 0.05 * np.random.randn(100)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg1 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg1.fit(X, y)

In [ ]:
# ?? ??
y2 = y - tree_reg1.predict(X)
tree_reg2 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg2.fit(X, y2)

In [ ]:
# ?? ??
y3 = y2 - tree_reg2.predict(X)
tree_reg3 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg3.fit(X, y3)

In [ ]:
X_new = np.array([[0.8]])
y_pred = sum(tree.predict(X_new) for tree in (tree_reg1, tree_reg2, tree_reg3))
y_pred

![Figure7-9](./images/Figure7-9.png)
**<center>?? 7-9 ????? ???</center>**

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=0.1, random_state=42)
gbrt.fit(X, y)

+ learning_rate : ? ??? ?? ??
+ learning_rate? 0.1?? ?? ???? ???? ?? ??? ????? ?? ?? ??? ????? ????? ??? ??? ???  
? **??**<sup>shrinkage</sup>?? ??? ?? ??

![Figure7-10](./images/Figure7-10.png)
**<center>?? 7-10 ???? ??? ??(??)? ?? ?? ??(???)? GBRT ???</center>**

+ ??? ?? ?? ?? ?? ? **?? ??** ??
+ staged_predict() : ??? ? ???? ???? ?? ???? ???? ???? ???<sup>iterator</sup>? ??

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=49)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120, random_state=42)
gbrt.fit(X_train, y_train)

errors = [mean_squared_error(y_val, y_pred)
          for y_pred in gbrt.staged_predict(X_val)]
bst_n_estimators = np.argmin(errors)

gbrt_best = GradientBoostingRegressor(max_depth=2,n_estimators=bst_n_estimators, random_state=42)
gbrt_best.fit(X_train, y_train)

![Figure7-11](./images/Figure7-11.png)
**<center>?? 7-11 ?? ??? ???? ?? ? ??<center>**

+ ?? ?? ??? ?? ????? ??? ?? ?? ?? ?? ???? ?? ??? ??? ???? ???? ?? ??? ??? ? ??
+ warm_start=True? ???? ????? fit() ???? ??? ? ?? ??? ???? ??? ??? ? ??? ??

In [ ]:
gbrt = GradientBoostingRegressor(max_depth=2, warm_start=True, random_state=42)

min_val_error = float("inf")
error_going_up = 0
for n_estimators in range(1, 120):
    gbrt.n_estimators = n_estimators
    gbrt.fit(X_train, y_train)
    y_pred = gbrt.predict(X_val)
    val_error = mean_squared_error(y_val, y_pred)
    if val_error < min_val_error:
        min_val_error = val_error
        error_going_up = 0
    else:
        error_going_up += 1
        if error_going_up == 5:
            break  # ?? ??

gbrt.n_estimators

+ ???? ?? ?? ?? ?? ?? ??? ???? ??? ??? ??

+ **??? ????? ???**<sup>Stochastic Gradient Boosting</sup>
    + subsample=0.25 ? ? ??? ???? ??? 25%? ?? ??? ???
    + ????? ??? ???? ?? ??? ???
    + ?? ??? ??? ??

In [ ]:
sto_gbrt = GradientBoostingRegressor(subsample=0.25, max_depth=2, n_estimators=3, learning_rate=0.1)
sto_gbrt.fit(X, y)

---
## 7.6 ???

+ **???**<sup>stacking</sup>? ?? ???? : ???? ?? ?? ???? ??? ???? ??? ??(?? ?? ??)? ???? ?? ???? ??? ???? ? ????
+ **???**<sup>blender</sup> ?? **?? ???**<sup>meta learner</sup>? ???? ???? ?? ?? ??? ?

![Figure7-12](./images/Figure7-12.png)
**<center>?? 7-12 ??? ???? ??? ?? ??<center>**

+ **???**<sup>blending</sup> : ?? ??<sup>hold-out</sup> ??? ??
+ **???**<sup>stacking</sup> : ?? ?? ??<sup>out-of-fold</sup> ??(k-? ?? ???? ?? ??? ??? ?? ?? ?)
+ ?? ??? ? ??? ?? ??? ??

![Figure7-13](./images/Figure7-13.png)
**<center>?? 7-13 ? ?? ??? ????<center>**

#### ? ??? ?? ??
1. ?? ??? ? ?? ????? ??
2. **? ?? ???**?? ????? ??
3. ? (? ?? ????) ????? **? ?? ???**(?? ?? ??)? ?? ??? ??
4. (???? ??? ??) ? ????? ? ?? ??(3??)? ?? ???? ??

![Figure7-14](./images/Figure7-14.png)
**<center>?? 7-14 ??? ??<center>**

#### ? ??(3)??? ??? ?? ??
1. ?? ??? ? ?? ????? ??
2. **? ?? ???**?? ????? ?? ? **? ?? ???**
3. ? ?? ???? ????? **? ?? ???**? ?? ??? ??
4. ? ????? ? ?? ??? ?? ????? ?? ? **? ?? ???**
5. ? ?? ???? ????? **? ?? ???**? ?? ??? ??
6. ? ????? ? ?? ??? ?? ???? ??

![Figure7-15](./images/Figure7-15.png)
**<center>?? 7-15 ?? ??? ??? ???? ??<center>**

---
## 7.7 ????

#### 1. ??? ?? ?? ???? ?? ?? ?? ??? ????? ?? 95% ???? ???? ? ???? ???? ? ?? ??? ?? ? ????? ????? ??? ?? ???? ??? ??? ?????

? ?? ?? ??? ????? ?? 95%? ???? ????? ??? ???? ?? ???? ??? ? ?? ??? ??? ? ????. ?? ??? ?? ???? (?? ?? SVM ???, ?? ?? ???, ???? ?? ??? ?) ?? ????. ?? ?? ?? ???? ?????? ??? ????(??? ??? ???? ???? ?????). ??? ??? ???? ??? ?? ?? ??? ??? ?? ??? ???.

#### 2. ?? ??? ?? ?? ??? ??? ???? ??????

? ?? ?? ???? ???? ?? ? ???? ??? ????? ?? ?? ??? ?? ???? ?????. ?? ?? ???? ? ???? ???? ?? ???? ???? ?? ?? ??? ?? ???? ????. ? ??? ??? ?? ??? ? ???? ?? ?? ? ?? ??? ???. ??? ???? ?? ?? ???? ??? ??? ??? ? ??? ??? ? ????(?? ?? ????? SVM ???? probability=True? ???? ???).

#### 3. ?? ???? ??? ?? ?? ??? ???? ??? ?? ? ????? ???? ???, ??? ???, ?? ????, ??? ???? ??? ?????

? ?? ???? ? ???? ?????? ?? ?? ??? ???? ???? ?? ??? ?? ? ????. ???? ???? ?? ????? ?? ??? ?????. ??? ??? ???? ???? ?? ???? ???? ?????? ??? ?????? ?? ?? ?? ??? ???? ?? ? ?? ??? ????. ??? ???? ?? ? ?? ?? ???? ?? ?????? ?? ?? ???? ??? ??? ? ????. ??? ? ?? ?? ????? ?? ?? ????? ??? ?? ??? ? ????.

#### 4. oob ??? ??? ??????

? oob ??? ???? ?? ???? ? ???? ??? ???? ??(?, ?? ????) ??? ??? ?????. ?? ???? ?? ??? ??? ???? ?? ???? ????? ?????. ???? ??? ? ?? ??? ??? ? ??? ???? ??? ?? ? ??? ????.

#### 5. ??? ???? ??? ?? ?? ?????? ? ????? ????? ???? ????? ??? ??? ???? ???? ??? ?? ?? ?????? ????, ?????

? ?? ?????? ??? ??? ? ? ???? ??? ??? ???? ??? ??? ?????. ???? ????? ?? ?????? ? ?? ? ???? ?? ?? ???? ??? ??? ???? ?? ?? ??? ? ??? ?? ??? ???? ?????. ? ???? ????? ???? ?????. ?, ?? ????? ?? ???? ???????? ???? ??? ??? ?? ????. ?? ???? ??? ??? ??? ???? ?? ?? ??? ?? ?????? ?? ??? ?????. ??? ??? ? ?? ?? ?????? ? ???? ???? ????.

#### 6. ????? ???? ?? ???? ???????? ?? ????? ??? ???? ????

? ????? ???? ?? ???? ???????? ??? ?? ?????? ?? ???? ?? ???????? ???? ? ? ????. ?? ???? ?? ???? ? ? ????.

#### 7. ????? ??? ???? ?? ???? ???????? ???? ??? ???, ??? ????

? ????? ??? ???? ?? ??? ???????? ???? ????? ???. (??? ?? ?? ???) ??? ??? ?? ?? ?? ?? ??? ??? ? ????.

#### 8. (3??? ???) MNIST ???? ???? ?? ??, ?? ??, ??? ??? ????(?? ?? ??? 40,000? ??, ??? 10,000? ??, ???? 10,000? ??). ?? ?? ?? ???? ???, ???? ?? ???, SVM ?? ?? ??? ???? ??????. ??? ?? ???? ??? ????? ? ?? ??? ??? ??? ?? ?? ?? ?? ???? ???? ???? ??????. ???? ?? ?? ??? ??? ??????. ??? ???? ???? ??? ??? ??????

In [ ]:
from sklearn.datasets import fetch_mldata

mnist = fetch_mldata('MNIST original')

In [ ]:
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(mnist.data, mnist.target, test_size=10000, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=10000, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

random_forest_clf = RandomForestClassifier(n_estimators=10, random_state=42)
extra_trees_clf = ExtraTreesClassifier(n_estimators=10, random_state=42)
svm_clf = LinearSVC(max_iter=10000, random_state=42)
mlp_clf = MLPClassifier(random_state=42)

estimators = [random_forest_clf, extra_trees_clf, svm_clf, mlp_clf]
for estimator in estimators:
    print("?? ???: ", estimator)
    estimator.fit(X_train, y_train)

In [ ]:
[estimator.score(X_val, y_val) for estimator in estimators]

In [ ]:
from sklearn.ensemble import VotingClassifier

named_estimators = [
    ("random_forest_clf", random_forest_clf),
    ("extra_trees_clf", extra_trees_clf),
    ("svm_clf", svm_clf),
    ("mlp_clf", mlp_clf),
]
voting_clf = VotingClassifier(named_estimators)
voting_clf.fit(X_train, y_train)

In [ ]:
voting_clf.score(X_val, y_val)

In [ ]:
[estimator.score(X_val, y_val) for estimator in voting_clf.estimators_]

In [ ]:
voting_clf.set_params(svm_clf=None)

In [ ]:
voting_clf.estimators

In [ ]:
voting_clf.estimators_

In [ ]:
del voting_clf.estimators_[2]

In [ ]:
voting_clf.score(X_val, y_val)

In [ ]:
voting_clf.voting = "soft"
voting_clf.score(X_val, y_val)

In [ ]:
voting_clf.score(X_test, y_test)

In [ ]:
[estimator.score(X_test, y_test) for estimator in voting_clf.estimators_]

#### 9. ?? ????? ? ???? ???? ?? ???? ??? ??? ? ??? ??? ?? ??? ??????. ? ?? ??? ??? ???? ?? ?? ???? ??? ?? ??? ??? ???? ??????. ??? ?? ??? ??? ??? ???????. ?? ???? ???????. ??? ? ???? ??? ??? ???? ??????. ?? ??? ??? ???? ??????. ??? ??? ? ???? ?? ?? ???? ??? ??? ???? ?? ??? ??? ?? ???? ? ??? ?????. ?? ?? ?? ???? ???? ?????

In [ ]:
X_val_predictions = np.empty((len(X_val), len(estimators)), dtype=np.float32)

for index, estimator in enumerate(estimators):
    X_val_predictions[:, index] = estimator.predict(X_val)

In [ ]:
X_val_predictions

In [ ]:
rnd_forest_blender = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)
rnd_forest_blender.fit(X_val_predictions, y_val)

In [ ]:
rnd_forest_blender.oob_score_

In [ ]:
X_test_predictions = np.empty((len(X_test), len(estimators)), dtype=np.float32)

for index, estimator in enumerate(estimators):
    X_test_predictions[:, index] = estimator.predict(X_test)

In [ ]:
y_pred = rnd_forest_blender.predict(X_test_predictions)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)